# 深大微堆假元件空间排布优化

本Notebook包含单方案OpenMC评价、离散遗传算法核心、mock回归测试、真实OpenMC screening pilot，以及正式20×15 GA和前10候选final复核接口。正式结果已经保存到独立目录；两个运行开关默认关闭，普通Run All不会重复启动OpenMC，也不包含控制棒插入工况。


## 1 优化问题与方案编码

每个方案由350个普通孔位中的5个假元件编号表示。编号顺序没有物理意义，因此先调用几何Notebook中的校验函数转换为排序元组。稳定目录标识使用SHA-256摘要的前12位，不使用跨进程可能变化的Python内置`hash`。

In [ ]:
import sys
from numbers import Integral, Real
from pathlib import Path
from datetime import datetime
from typing import cast
import contextlib
import hashlib
import io
import json
import math
import os
import time
import xml.etree.ElementTree as ET

import h5py
import matplotlib.pyplot as plt
import numpy as np
import openmc
import pandas as pd
from IPython import get_ipython
from IPython.display import display


def _execute_notebook_code(source, label):
    result = get_ipython().run_cell(source)
    if result.error_before_exec is not None:
        raise RuntimeError(f"加载{label}时发生执行前错误。") from result.error_before_exec
    if result.error_in_exec is not None:
        raise RuntimeError(f"加载{label}时发生运行错误。") from result.error_in_exec


materials_notebook_path = Path("MNSR_materials_development.ipynb")
geometry_notebook_path = Path("MNSR_geometry_development.ipynb")
for notebook_path in (materials_notebook_path, geometry_notebook_path):
    if not notebook_path.is_file():
        raise FileNotFoundError(f"找不到已有Notebook：{notebook_path.resolve()}")

# 只加载材料与几何定义，跳过绘图、XML导出和参考方案构建展示。
materials_notebook = json.loads(materials_notebook_path.read_text(encoding="utf-8"))
for cell_index, cell in enumerate(materials_notebook["cells"]):
    if cell["cell_type"] == "code" and cell_index < 31:
        _execute_notebook_code("".join(cell["source"]), f"材料Notebook第{cell_index}单元")

geometry_notebook = json.loads(geometry_notebook_path.read_text(encoding="utf-8"))
geometry_skipped_cells = {3, 42, 44, 48, 50, 56, 58}
for cell_index, cell in enumerate(geometry_notebook["cells"]):
    if cell["cell_type"] != "code" or cell_index in geometry_skipped_cells:
        continue
    source = "".join(cell["source"])
    if cell_index == 20:
        source = source.split("figure, axis", 1)[0]
    elif cell_index == 46:
        source = source.split("POSITION_TABLE_PATH", 1)[0]
    elif cell_index == 54:
        source = source.split("global_component_types_before", 1)[0]
    _execute_notebook_code(source, f"几何Notebook第{cell_index}单元")

required_objects = {
    "materials", "material_map", "ordinary_positions_df", "position_table",
    "build_geometry_for_scheme", "validate_dummy_position_ids", "REFERENCE_DUMMY_IDS",
}
missing_objects = required_objects - set(globals())
assert not missing_objects, f"缺少几何对象：{sorted(missing_objects)}"

print("Python解释器：", sys.executable)
print("OpenMC版本：", openmc.__version__)
print("IAEA参考几何与材料定义加载成功。")


In [ ]:
def canonicalize_scheme(dummy_position_ids):
    """校验假元件编号，并返回排序后可稳定比较的方案元组。"""
    allowed_position_ids = set(
        int(position_id)
        for position_id in ordinary_positions_df["ordinary_position_id"]
    )
    return validate_dummy_position_ids(
        dummy_position_ids,
        allowed_position_ids=allowed_position_ids,
    )


def scheme_key(dummy_position_ids):
    """用规范方案字符串的SHA-256前12位生成稳定目录标识。"""
    canonical_ids = canonicalize_scheme(dummy_position_ids)
    scheme_text = ",".join(str(position_id) for position_id in canonical_ids)
    return hashlib.sha256(scheme_text.encode("utf-8")).hexdigest()[:12]


assert canonicalize_scheme({337, 285, 324, 298, 311}) == REFERENCE_DUMMY_IDS
assert scheme_key(REFERENCE_DUMMY_IDS) == scheme_key(tuple(reversed(REFERENCE_DUMMY_IDS)))

print("参考方案：", canonicalize_scheme(REFERENCE_DUMMY_IDS))
print("参考方案稳定标识：", scheme_key(REFERENCE_DUMMY_IDS))

## 2 OpenMC计算参数

`debug`用于流程检查，`screening`用于真实GA初筛，`final`只用于少量候选复核。所有真实计算均受显式开关和独立调用上限保护，并使用完整cache identity；不能由12位方案ID单独判定缓存命中。


In [ ]:
RUN_PROFILES = {
    "debug": {
        "particles": 1000,
        "batches": 20,
        "inactive": 5,
    },
    "screening": {
        "particles": 5000,
        "batches": 60,
        "inactive": 15,
    },
    "final": {
        "particles": 20000,
        "batches": 150,
        "inactive": 50,
    },
}

display(pd.DataFrame(RUN_PROFILES).T)

### Settings创建函数

本征值源只在堆芯燃料区域附近的方盒中抽样，并通过`fissionable`约束拒绝非裂变材料位置，避免把源均匀撒到整个容器。温度采用与293.15 K材料模型兼容的最近温度处理。随机种子由方案标识稳定派生，使同一方案可复现。

In [ ]:
def create_settings(profile_name, random_seed):
    """按指定配置创建可复现的OpenMC本征值计算设置。"""
    if profile_name not in RUN_PROFILES:
        raise ValueError(
            f"未知计算配置{profile_name!r}，可选值为{list(RUN_PROFILES)}。"
        )
    if isinstance(random_seed, bool) or not isinstance(random_seed, int):
        raise ValueError("random_seed必须是正整数。")
    if random_seed <= 0:
        raise ValueError("random_seed必须大于0。")

    profile = RUN_PROFILES[profile_name]
    settings = openmc.Settings()
    settings.run_mode = "eigenvalue"
    settings.particles = profile["particles"]
    settings.batches = profile["batches"]
    settings.inactive = profile["inactive"]
    settings.seed = random_seed
    settings.temperature = {
        "method": "nearest",
        "tolerance": 100.0,
    }

    source_space = openmc.stats.Box(
        (-11.25, -11.25, Z_BOTTOM + 1.0e-6),
        (11.25, 11.25, Z_TOP - 1.0e-6),
    )
    settings.source = openmc.IndependentSource(
        space=source_space,
        constraints={"fissionable": True},
    )
    return settings

## 3 功率计数器设置

优化评价只对345个燃料芯体Cell建立`CellFilter`，且严格保留几何metadata中的Cell顺序。唯一score为`kappa-fission`，它表示单位源粒子归一化的裂变能释放计数，可用于比较棒间相对功率；它不是热工计算得到的燃料温度。本流程不创建MeshFilter或完整三维通量图。

In [ ]:
def create_power_tally(fuel_meat_cells):
    """为有序燃料芯体Cell创建kappa-fission Tally及孔位映射。"""
    ordered_cells = list(fuel_meat_cells)
    if len(ordered_cells) != 345:
        raise ValueError(
            f"功率Tally需要345个燃料芯体Cell，当前为{len(ordered_cells)}。"
        )

    fuel_cell_position_ids = []
    for cell in ordered_cells:
        prefix = "fuel_meat_pos_"
        if not cell.name.startswith(prefix):
            raise ValueError(f"无法从Cell名称读取普通孔位编号：{cell.name}")
        fuel_cell_position_ids.append(int(cell.name.removeprefix(prefix)))

    cell_filter = openmc.CellFilter(ordered_cells)
    power_tally = openmc.Tally(name="fuel_pin_kappa_fission")
    power_tally.filters = [cell_filter]
    power_tally.scores = ["kappa-fission"]

    tallies = openmc.Tallies([power_tally])
    return tallies, tuple(fuel_cell_position_ids)


## 4 单方案计算目录

每个规范方案拥有独立目录`build/optimization/cases/{scheme_key}/`，保存方案说明、四个OpenMC输入XML、运行输出、棒功率表和结果JSON。不同方案互不覆盖；无法规范化的非法输入则写入`invalid_*`错误目录。

In [ ]:
OPTIMIZATION_ROOT = Path("build/optimization").resolve()
CASES_ROOT = OPTIMIZATION_ROOT / "cases"
CASES_ROOT.mkdir(parents=True, exist_ok=True)

print("优化开发输出根目录：", OPTIMIZATION_ROOT)
# 默认安全关闭；只有显式环境变量才允许pilot、正式screening或final调用OpenMC。
ALLOW_REAL_OPENMC = False
RUN_REAL_GA_PILOT = False
RUN_FORMAL_REAL_GA = os.environ.get("MNSR_RUN_FORMAL_GA", "0") == "1"
RUN_FINALIST_REVIEW = os.environ.get("MNSR_RUN_FINALISTS", "0") == "1"
RUN_STATISTICAL_CONFIRMATION = os.environ.get("MNSR_RUN_STAT_CONFIRMATION", "0") == "1"



## 5 单方案评价函数

`evaluate_scheme`先检查缓存，再构建withdrawn几何、Settings和功率Tally，并用`openmc.Model`统一管理。`run_openmc=False`只导出输入并返回`exported`，所有物理结果保持`None`。真实运行后，反应性按ρ=(k−1)/k计算；一阶误差传播使用dρ/dk=1/k²，因此σρ≈σk/k²。棒功率峰值因子和变异系数直接使用单位源粒子归一化的`kappa-fission`，不需要先换算到30 kW。

In [ ]:
def _write_json(path, payload):
    """以UTF-8和缩进格式保存教学流程中的JSON文件。"""
    path.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )


def _prepare_evaluation_materials(
    scheme_geometry,
    cross_sections_path,
):
    """为当前截面库建立不修改全局材料对象的评价材料集合。"""
    library_root = ET.parse(cross_sections_path).getroot()
    thermal_table_names = {
        library.get("materials")
        for library in library_root.findall("library")
        if library.get("type") == "thermal"
    }

    requested_table = "c_Be_in_Be"
    if requested_table in thermal_table_names:
        return materials, {}

    available_metal_be_table = "c_Be"
    if available_metal_be_table not in thermal_table_names:
        raise FileNotFoundError(
            "当前截面库既不含c_Be_in_Be，也不含可兼容的金属铍c_Be热散射表。"
        )

    original_beryllium = material_map["beryllium"]
    evaluation_beryllium = openmc.Material(
        name=original_beryllium.name
    )
    for nuclide in original_beryllium.nuclides:
        evaluation_beryllium.add_nuclide(
            nuclide.name,
            nuclide.percent,
            nuclide.percent_type,
        )

    if original_beryllium.density is None:
        raise ValueError("原金属铍材料尚未设置密度。")
    evaluation_beryllium.set_density(
        original_beryllium.density_units,
        original_beryllium.density,
    )
    if original_beryllium.temperature is not None:
        evaluation_beryllium.temperature = cast(
            Real,
            original_beryllium.temperature,
        )
    evaluation_beryllium.add_s_alpha_beta(available_metal_be_table)

    replaced_cells = 0
    for cell in scheme_geometry.get_all_cells().values():
        if cell.fill is original_beryllium:
            cell.fill = evaluation_beryllium
            replaced_cells += 1
    if replaced_cells != 1:
        raise RuntimeError(
            f"应替换1个侧铍Cell，实际替换{replaced_cells}个。"
        )

    evaluation_materials = openmc.Materials([
        evaluation_beryllium
        if material is original_beryllium
        else material
        for material in materials
    ])
    aliases = {
        requested_table: available_metal_be_table,
    }
    return evaluation_materials, aliases


def evaluate_scheme(
    dummy_position_ids,
    profile_name="debug",
    force_rerun=False,
    run_openmc=True,
):
    """导出或运行一个假元件方案，并返回可缓存的评价结果字典。"""
    case_directory = None
    normalized_ids = None
    current_scheme_key = None

    try:
        normalized_ids = canonicalize_scheme(dummy_position_ids)
        current_scheme_key = scheme_key(normalized_ids)
        case_directory = (CASES_ROOT / current_scheme_key).resolve()
        case_directory.mkdir(parents=True, exist_ok=True)

        if profile_name not in RUN_PROFILES:
            raise ValueError(
                f"未知计算配置{profile_name!r}，可选值为{list(RUN_PROFILES)}。"
            )
        if profile_name in {"screening", "final"}:
            real_evaluator = globals().get("evaluate_scheme_real")
            if real_evaluator is None:
                raise RuntimeError("请先执行真实GA评价接口定义单元。")
            return real_evaluator(
                dummy_position_ids,
                profile_name=profile_name,
                force_rerun=force_rerun,
                run_openmc=run_openmc,
                allow_real_openmc=ALLOW_REAL_OPENMC,
            )
        if not isinstance(force_rerun, bool) or not isinstance(run_openmc, bool):
            raise ValueError("force_rerun和run_openmc必须是布尔值。")

        result_path = case_directory / "result.json"
        if result_path.is_file() and not force_rerun:
            return json.loads(result_path.read_text(encoding="utf-8"))
        if force_rerun:
            (case_directory / "error.json").unlink(missing_ok=True)

        try:
            cross_sections_value = openmc.config["cross_sections"]
        except KeyError as error:
            raise FileNotFoundError("尚未设置OpenMC截面库路径。") from error
        cross_sections_path = Path(cross_sections_value).expanduser().resolve()
        if not cross_sections_path.is_file():
            raise FileNotFoundError(
                f"OpenMC截面库文件不存在：{cross_sections_path}"
            )

        random_seed = int(current_scheme_key, 16) % 2_147_483_646 + 1
        scheme_description = {
            "scheme_key": current_scheme_key,
            "dummy_position_ids": list(normalized_ids),
            "profile_name": profile_name,
            "random_seed": random_seed,
            "cross_sections": str(cross_sections_path),
        }
        _write_json(case_directory / "scheme.json", scheme_description)

        try:
            scheme_geometry, geometry_metadata = build_geometry_for_scheme(
                normalized_ids,
                control_rod_state="withdrawn",
                export_directory=None,
                create_plots=False,
            )
        except Exception as error:
            raise RuntimeError(f"方案几何构建失败：{error}") from error

        evaluation_materials, thermal_scattering_aliases = (
            _prepare_evaluation_materials(
                scheme_geometry,
                cross_sections_path,
            )
        )
        scheme_description["thermal_scattering_aliases"] = (
            thermal_scattering_aliases
        )
        _write_json(case_directory / "scheme.json", scheme_description)

        settings = create_settings(profile_name, random_seed)
        tallies, fuel_cell_position_ids = create_power_tally(
            geometry_metadata["fuel_meat_cells"]
        )

        model = openmc.Model()
        model.materials = evaluation_materials
        model.geometry = scheme_geometry
        model.settings = settings
        model.tallies = tallies
        model.export_to_xml(directory=case_directory)

        required_input_files = {
            "scheme.json",
            "materials.xml",
            "geometry.xml",
            "settings.xml",
            "tallies.xml",
        }
        missing_files = sorted(
            file_name
            for file_name in required_input_files
            if not (case_directory / file_name).is_file()
        )
        if missing_files:
            raise RuntimeError(f"方案输入文件导出不完整：{missing_files}")

        result = {
            "scheme_key": current_scheme_key,
            "dummy_position_ids": list(normalized_ids),
            "profile_name": profile_name,
            "keff_mean": None,
            "keff_std": None,
            "reactivity": None,
            "reactivity_std_approx": None,
            "mean_pin_power": None,
            "max_pin_power": None,
            "min_pin_power": None,
            "radial_peaking_factor": None,
            "power_coefficient_of_variation": None,
            "max_power_position_id": None,
            "run_directory": str(case_directory),
            "thermal_scattering_aliases": thermal_scattering_aliases,
            "status": "exported",
        }

        if not run_openmc:
            _write_json(result_path, result)
            return result

        executable_candidates = [
            Path(sys.executable).with_name("openmc"),
            Path(sys.prefix) / "envs" / "openmc-env" / "bin" / "openmc",
        ]
        openmc_executable = next(
            (path for path in executable_candidates if path.is_file()),
            None,
        )
        if openmc_executable is None:
            raise FileNotFoundError("找不到OpenMC可执行程序，无法运行方案。")

        from unittest.mock import PropertyMock, patch

        try:
            with patch.object(
                openmc.Model,
                "is_initialized",
                new_callable=PropertyMock,
                return_value=False,
            ):
                statepoint_path = model.run(
                    cwd=case_directory,
                    openmc_exec=openmc_executable,
                    export_model_xml=False,
                )
        except Exception as error:
            raise RuntimeError(
                f"OpenMC运行失败或返回码非零：{error}"
            ) from error

        if statepoint_path is None:
            statepoint_candidates = sorted(
                case_directory.glob("statepoint.*.h5")
            )
            if not statepoint_candidates:
                raise FileNotFoundError("OpenMC运行后未找到statepoint文件。")
            statepoint_path = statepoint_candidates[-1]
        else:
            statepoint_path = Path(statepoint_path)
            if not statepoint_path.is_absolute():
                statepoint_path = case_directory / statepoint_path
        if not statepoint_path.is_file():
            raise FileNotFoundError(f"statepoint文件不存在：{statepoint_path}")

        try:
            with openmc.StatePoint(statepoint_path) as statepoint:
                keff = statepoint.keff
                keff_mean = float(keff.nominal_value)
                keff_std = float(keff.std_dev)
                power_tally = statepoint.get_tally(
                    name="fuel_pin_kappa_fission"
                )
                pin_values = np.asarray(power_tally.mean, dtype=float).reshape(-1)
                pin_std_values = np.asarray(
                    power_tally.std_dev,
                    dtype=float,
                ).reshape(-1)
        except Exception as error:
            raise RuntimeError(
                f"无法从statepoint读取keff或功率Tally：{error}"
            ) from error

        if pin_values.size != 345 or pin_std_values.size != 345:
            raise ValueError(
                "功率Tally维数错误："
                f"mean={pin_values.size}, std={pin_std_values.size}，应为345。"
            )
        if not np.all(np.isfinite(pin_values)):
            raise ValueError("kappa-fission均值包含非有限值。")
        if not np.all(np.isfinite(pin_std_values)):
            raise ValueError("kappa-fission标准差包含非有限值。")

        mean_pin_power = float(np.mean(pin_values))
        if not math.isfinite(mean_pin_power) or mean_pin_power <= 0.0:
            raise ValueError("kappa-fission平均值为零、负值或非有限值。")

        max_index = int(np.argmax(pin_values))
        max_power_position_id = int(fuel_cell_position_ids[max_index])
        max_pin_power = float(np.max(pin_values))
        min_pin_power = float(np.min(pin_values))
        radial_peaking_factor = max_pin_power / mean_pin_power
        power_coefficient_of_variation = (
            float(np.std(pin_values, ddof=0)) / mean_pin_power
        )
        reactivity = (keff_mean - 1.0) / keff_mean
        reactivity_std_approx = keff_std / (keff_mean ** 2)

        pin_power_table = pd.DataFrame({
            "ordinary_position_id": fuel_cell_position_ids,
            "cell_id": [
                cell.id for cell in geometry_metadata["fuel_meat_cells"]
            ],
            "kappa_fission_mean": pin_values,
            "kappa_fission_std": pin_std_values,
            "relative_power": pin_values / mean_pin_power,
        })
        pin_power_path = case_directory / "pin_power.csv"
        pin_power_table.to_csv(pin_power_path, index=False, encoding="utf-8")

        result.update({
            "keff_mean": keff_mean,
            "keff_std": keff_std,
            "reactivity": float(reactivity),
            "reactivity_std_approx": float(reactivity_std_approx),
            "mean_pin_power": mean_pin_power,
            "max_pin_power": max_pin_power,
            "min_pin_power": min_pin_power,
            "radial_peaking_factor": float(radial_peaking_factor),
            "power_coefficient_of_variation": float(
                power_coefficient_of_variation
            ),
            "max_power_position_id": max_power_position_id,
            "statepoint_file": str(statepoint_path.resolve()),
            "pin_power_file": str(pin_power_path.resolve()),
            "status": "completed",
        })
        _write_json(result_path, result)
        return result

    except Exception as error:
        if case_directory is None:
            invalid_key = hashlib.sha256(
                repr(dummy_position_ids).encode("utf-8")
            ).hexdigest()[:12]
            case_directory = CASES_ROOT / f"invalid_{invalid_key}"
            case_directory.mkdir(parents=True, exist_ok=True)

        error_payload = {
            "scheme_key": current_scheme_key,
            "dummy_position_ids": (
                list(normalized_ids) if normalized_ids is not None else None
            ),
            "profile_name": profile_name,
            "error_type": type(error).__name__,
            "error_message": str(error),
            "timestamp": datetime.now().isoformat(),
        }
        _write_json(case_directory / "error.json", error_payload)
        raise RuntimeError(
            f"方案评价失败，错误信息已保存到"
            f"{case_directory / 'error.json'}：{error}"
        ) from error

## 6 参考排布试运行

第一步只导出参考方案，不启动OpenMC。只有确认方案目录和全部输入文件正确后，下一单元才强制运行一次debug方案。本轮不会运行第二个真实方案；如OpenMC失败，Notebook立即停止并保留输入与`error.json`。

In [ ]:
reference_export = evaluate_scheme(
    REFERENCE_DUMMY_IDS,
    profile_name="debug",
    force_rerun=False,
    run_openmc=False,
)

reference_case_directory = Path(reference_export["run_directory"])
expected_export_files = {
    "scheme.json",
    "materials.xml",
    "geometry.xml",
    "settings.xml",
    "tallies.xml",
    "result.json",
}
assert reference_export["status"] in {"exported", "completed"}
assert all(
    (reference_case_directory / file_name).is_file()
    for file_name in expected_export_files
)

print("参考方案输入导出检查通过：", reference_case_directory)

In [ ]:
# 默认不重复运行旧debug算例；已有结果仅作为教学流程缓存读取。
RUN_REFERENCE_DEBUG = False
reference_result = evaluate_scheme(
    REFERENCE_DUMMY_IDS,
    profile_name="debug",
    force_rerun=RUN_REFERENCE_DEBUG,
    run_openmc=RUN_REFERENCE_DEBUG,
)
assert reference_result["status"] in {"exported", "completed"}
print("参考debug流程状态：", reference_result["status"])
print("本次Run All是否重新运行debug OpenMC：", RUN_REFERENCE_DEBUG)


## 7 评价结果检查

这里展示唯一debug方案的keff、反应性、棒功率峰值因子和变异系数，并将345根棒的单位源粒子相对功率画在普通孔位坐标上。图题明确标注低统计量，仅用于程序流程检查，不作为论文正式结论。

In [ ]:
print(
    f"keff = {reference_result['keff_mean']:.7f} "
    f"± {reference_result['keff_std']:.7f}"
)
print(
    f"反应性ρ = {reference_result['reactivity']:.7e} "
    f"± {reference_result['reactivity_std_approx']:.7e}（一阶近似）"
)
print("径向功率峰值因子：", reference_result["radial_peaking_factor"])
print("功率变异系数：", reference_result["power_coefficient_of_variation"])
print("最大棒功率普通孔位：", reference_result["max_power_position_id"])

pin_power_table = pd.read_csv(reference_result["pin_power_file"])
assert len(pin_power_table) == 345
plot_table = pin_power_table.merge(
    ordinary_positions_df[["ordinary_position_id", "x_cm", "y_cm"]],
    on="ordinary_position_id",
    how="left",
    validate="one_to_one",
)

figure, axis = plt.subplots(figsize=(9, 8))
scatter = axis.scatter(
    plot_table["x_cm"],
    plot_table["y_cm"],
    c=plot_table["relative_power"],
    s=32,
    cmap="viridis",
)
axis.set_aspect("equal")
axis.set_xlabel("x [cm]", fontsize=13)
axis.set_ylabel("y [cm]", fontsize=13)
axis.set_title(
    "345根燃料棒归一化相对功率\n"
    "debug低统计量计算，仅用于程序流程检查",
    fontsize=15,
)
colorbar = figure.colorbar(scatter, ax=axis)
colorbar.set_label("相对平均棒功率")

power_plot_path = (
    Path(reference_result["run_directory"])
    / "normalized_pin_power.png"
)
figure.savefig(power_plot_path, dpi=180, bbox_inches="tight")
plt.show()

assert Path(reference_result["statepoint_file"]).is_file()
assert Path(reference_result["pin_power_file"]).is_file()
assert power_plot_path.is_file()

print("结果文件：", Path(reference_result["run_directory"]))
print("单方案评价流程检查完成；未启动遗传算法或批量方案运行。")

## 8 真实遗传算法核心与安全配置

个体始终表示为5个升序、唯一、合法普通孔位的元组。外部输入使用严格校验；`repair_individual`只修复交叉或变异产生的中间个体。真实GA与旧mock目录完全隔离，默认安全开关为关闭状态。


In [ ]:
REAL_GA_ALGORITHM_VERSION = "discrete_set_ga_v2"
REAL_GA_GEOMETRY_VERSION = "iaea_reference_v1"
REAL_GA_MODEL_VARIANT = "iaea_reference"
REAL_GA_CONTROL_ROD_STATE = "withdrawn"
REAL_GA_MATERIAL_VERSION = "tecdoc1844_leu_materials_v1"
REAL_GA_TALLY_VERSION = "fuel_pin_kappa_fission_v1"
REAL_GA_SEED = 20260728
OPENMC_BASE_SEED = 20260730
FAILURE_PENALTY = 1.0e6
MAX_NEW_SCREENING_OPENMC_CASES = 10
MAX_FORMAL_NEW_SCREENING_OPENMC_CASES = 280
MAX_NEW_FINAL_OPENMC_CASES = 10

REAL_GA_ROOT = Path("build/optimization/real_ga").resolve()
REAL_GA_SCHEMES_ROOT = REAL_GA_ROOT / "schemes"
REAL_GA_CHECKPOINT_ROOT = REAL_GA_ROOT / "checkpoints"
REAL_GA_TEST_ROOT = REAL_GA_ROOT / "tests"
REAL_GA_PILOT_ROOT = REAL_GA_ROOT / "pilot"
for directory in (REAL_GA_SCHEMES_ROOT, REAL_GA_CHECKPOINT_ROOT, REAL_GA_TEST_ROOT, REAL_GA_PILOT_ROOT):
    directory.mkdir(parents=True, exist_ok=True)

ALLOWED_POSITION_IDS = tuple(
    int(value) for value in ordinary_positions_df["ordinary_position_id"].tolist()
)
ALLOWED_POSITION_ID_SET = frozenset(ALLOWED_POSITION_IDS)
if ALLOWED_POSITION_IDS != tuple(range(350)):
    raise AssertionError("普通候选孔位必须恰好为0至349。")


def normalize_individual(candidate):
    """只做确定性排序和整数规范化，不替代严格合法性检查。"""
    try:
        values = list(candidate)
    except TypeError as error:
        raise ValueError("个体必须是可迭代的5个整数孔位。") from error
    if any(isinstance(value, bool) or not isinstance(value, Integral) for value in values):
        raise ValueError("个体中的每个孔位必须是整数且不能是布尔值。")
    return tuple(sorted(int(value) for value in values))


def validate_individual(candidate, allowed_position_ids=None):
    """严格校验外部方案并返回规范5元组；失败时绝不静默repair。"""
    allowed = ALLOWED_POSITION_ID_SET if allowed_position_ids is None else frozenset(int(x) for x in allowed_position_ids)
    normalized = normalize_individual(candidate)
    if len(normalized) != 5:
        raise ValueError(f"个体必须恰好包含5个孔位，当前为{len(normalized)}个。")
    if len(set(normalized)) != 5:
        raise ValueError("个体孔位必须互不重复。")
    invalid = [value for value in normalized if value not in allowed]
    if invalid:
        raise ValueError(f"个体包含不允许的普通孔位：{invalid}")
    return normalized


def repair_individual(candidate, rng, allowed_position_ids=None):
    """修复遗传操作中间个体，补足或裁剪为合法升序5元组。"""
    if not isinstance(rng, np.random.Generator):
        raise TypeError("rng必须是numpy.random.Generator。")
    allowed = tuple(ALLOWED_POSITION_IDS if allowed_position_ids is None else sorted(set(int(x) for x in allowed_position_ids)))
    allowed_set = frozenset(allowed)
    retained = []
    for value in list(candidate):
        if isinstance(value, bool) or not isinstance(value, Integral):
            continue
        integer = int(value)
        if integer in allowed_set and integer not in retained:
            retained.append(integer)
    if len(retained) > 5:
        retained = list(rng.choice(np.asarray(retained, dtype=int), size=5, replace=False))
    missing = 5 - len(retained)
    if missing > 0:
        available = np.asarray(sorted(allowed_set - set(retained)), dtype=int)
        if available.size < missing:
            raise ValueError("候选空间不足以repair为5个唯一孔位。")
        retained.extend(int(x) for x in rng.choice(available, size=missing, replace=False))
    return validate_individual(retained, allowed)


def create_random_individual(rng, allowed_position_ids=None):
    allowed = np.asarray(ALLOWED_POSITION_IDS if allowed_position_ids is None else sorted(set(int(x) for x in allowed_position_ids)), dtype=int)
    if allowed.size < 5:
        raise ValueError("候选孔位少于5个，无法生成个体。")
    return validate_individual(rng.choice(allowed, size=5, replace=False), allowed)


def initialize_population(population_size, reference_scheme, rng, allowed_position_ids=None):
    """建立唯一初始种群，并把合法参考方案固定放在第0位。"""
    if isinstance(population_size, bool) or not isinstance(population_size, Integral) or population_size < 1:
        raise ValueError("population_size必须为正整数。")
    allowed = tuple(ALLOWED_POSITION_IDS if allowed_position_ids is None else sorted(set(int(x) for x in allowed_position_ids)))
    reference = validate_individual(reference_scheme, allowed)
    if math.comb(len(allowed), 5) < int(population_size):
        raise ValueError("候选组合空间不足以生成指定数量的唯一个体。")
    population = [reference]
    seen = {reference}
    while len(population) < int(population_size):
        individual = create_random_individual(rng, allowed)
        if individual not in seen:
            seen.add(individual)
            population.append(individual)
    return population


def tournament_select(population, fitness_records, tournament_size, rng):
    """最小化目标的锦标赛选择。"""
    if tournament_size < 1 or tournament_size > len(population):
        raise ValueError("tournament_size必须位于1和种群大小之间。")
    indices = rng.choice(len(population), size=int(tournament_size), replace=False)
    candidates = [population[int(index)] for index in indices]
    return min(candidates, key=lambda individual: float(fitness_records[individual]["fitness"]))


def set_crossover(parent_a, parent_b, rng, allowed_position_ids=None):
    """集合交叉：保留交集，从父代并集和候选空间补足两个子代。"""
    allowed = tuple(ALLOWED_POSITION_IDS if allowed_position_ids is None else sorted(set(int(x) for x in allowed_position_ids)))
    a = validate_individual(parent_a, allowed)
    b = validate_individual(parent_b, allowed)
    common = sorted(set(a) & set(b))
    union_extra = sorted((set(a) | set(b)) - set(common))

    def make_child():
        candidate = list(common)
        if union_extra:
            order = rng.permutation(np.asarray(union_extra, dtype=int))
            candidate.extend(int(x) for x in order[: max(0, 5 - len(candidate))])
        return repair_individual(candidate, rng, allowed)

    return make_child(), make_child()


def replace_one_mutation(individual, mutation_probability, rng, allowed_position_ids=None):
    """按概率以一个未占用孔位替换一个现有孔位。"""
    if not 0.0 <= float(mutation_probability) <= 1.0:
        raise ValueError("mutation_probability必须位于0和1之间。")
    allowed = tuple(ALLOWED_POSITION_IDS if allowed_position_ids is None else sorted(set(int(x) for x in allowed_position_ids)))
    normalized = validate_individual(individual, allowed)
    if rng.random() >= float(mutation_probability):
        return normalized
    available = np.asarray(sorted(set(allowed) - set(normalized)), dtype=int)
    if available.size == 0:
        return normalized
    mutated = list(normalized)
    replace_index = int(rng.integers(0, 5))
    mutated[replace_index] = int(rng.choice(available))
    return validate_individual(mutated, allowed)


def compute_fitness(result, constraints=None):
    """论文screening目标：成功方案fitness=Fr，失败方案加有限大惩罚。"""
    constraints = {} if constraints is None else dict(constraints)
    rho_min = constraints.get("rho_min")
    rho_max = constraints.get("rho_max")
    penalty_weight = float(constraints.get("penalty_weight", 10000.0))
    failure_penalty = float(constraints.get("failure_penalty", FAILURE_PENALTY))
    status = result.get("status")
    if status not in {"completed", "mock_completed"}:
        return {"fitness": failure_penalty, "reactivity_penalty": 0.0, "failure_penalty": failure_penalty}
    fr = float(result["radial_peaking_factor"])
    rho = float(result.get("reactivity", result.get("rho", 0.0)))
    if not math.isfinite(fr) or fr <= 0.0:
        return {"fitness": failure_penalty, "reactivity_penalty": 0.0, "failure_penalty": failure_penalty}
    reactivity_penalty = 0.0
    if rho_min is not None and rho < float(rho_min):
        reactivity_penalty += penalty_weight * (float(rho_min) - rho) ** 2
    if rho_max is not None and rho > float(rho_max):
        reactivity_penalty += penalty_weight * (rho - float(rho_max)) ** 2
    return {"fitness": fr + reactivity_penalty, "reactivity_penalty": reactivity_penalty, "failure_penalty": 0.0}


def mock_evaluate_scheme(individual):
    """确定性算法回归目标，不代表OpenMC物理结果。"""
    scheme = validate_individual(individual)
    rows = ordinary_positions_df.set_index("ordinary_position_id").loc[list(scheme)]
    angles = np.arctan2(rows["y_cm"].to_numpy(float), rows["x_cm"].to_numpy(float))
    angular_resultant = abs(np.mean(np.exp(1j * angles)))
    xy = rows[["x_cm", "y_cm"]].to_numpy(float)
    distances = [float(np.linalg.norm(xy[i] - xy[j])) for i in range(5) for j in range(i + 1, 5)]
    proximity_penalty = sum(max(0.0, 3.0 - distance) ** 2 for distance in distances) / 25.0
    objective = 1.0 + 0.45 * angular_resultant + 0.05 * proximity_penalty
    return {
        "scheme_id": scheme_key(scheme), "normalized_dummy_ids": list(scheme),
        "profile": "mock", "keff": None, "keff_std": None, "reactivity": 0.0,
        "rho_mk": 0.0, "radial_peaking_factor": float(objective),
        "power_coefficient_of_variation": None, "max_power_position_id": None,
        "mean_pin_relative_uncertainty": None, "calculation_seconds": 0.0,
        "cache_hit": False, "status": "mock_completed",
    }


def create_next_generation(population, fitness_records, population_size, elite_size, crossover_probability, mutation_probability, tournament_size, rng, allowed_position_ids=None):
    """精英保留、锦标赛、集合交叉和单孔位变异生成唯一下一代。"""
    if not 1 <= int(elite_size) < int(population_size):
        raise ValueError("elite_size必须至少为1且小于population_size。")
    allowed = tuple(ALLOWED_POSITION_IDS if allowed_position_ids is None else sorted(set(int(x) for x in allowed_position_ids)))
    ranked = sorted(population, key=lambda ind: float(fitness_records[ind]["fitness"]))
    next_population = list(ranked[: int(elite_size)])
    seen = set(next_population)
    attempts = 0
    while len(next_population) < int(population_size):
        attempts += 1
        if attempts > 10000:
            raise RuntimeError("生成唯一下一代超过10000次尝试。")
        parent_a = tournament_select(population, fitness_records, tournament_size, rng)
        parent_b = tournament_select(population, fitness_records, tournament_size, rng)
        if rng.random() < float(crossover_probability):
            children = set_crossover(parent_a, parent_b, rng, allowed)
        else:
            children = (parent_a, parent_b)
        for child in children:
            child = replace_one_mutation(child, mutation_probability, rng, allowed)
            child = repair_individual(child, rng, allowed)
            if child not in seen:
                seen.add(child)
                next_population.append(child)
                if len(next_population) == int(population_size):
                    break
    return next_population


## 9 Checkpoint与遗传算法主循环

checkpoint保存配置、下一代编号、种群、RNG状态、历史、评价缓存和失败记录。resume前逐字段验证配置，真实GA绝不读取旧mock checkpoint。


In [ ]:
def _jsonable(value):
    if isinstance(value, dict):
        return {str(key): _jsonable(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_jsonable(item) for item in value]
    if isinstance(value, np.generic):
        return value.item()
    return value


def validate_checkpoint_config(checkpoint_config, expected_config):
    required = (
        "candidate_position_count", "crossover_probability", "mutation_probability",
        "tournament_size", "ga_seed", "population_size", "generations", "elite_size",
        "geometry_model_version", "screening_profile", "algorithm_version",
    )
    mismatches = {
        key: {"checkpoint": checkpoint_config.get(key), "expected": expected_config.get(key)}
        for key in required if checkpoint_config.get(key) != expected_config.get(key)
    }
    if mismatches:
        raise ValueError(f"checkpoint配置不一致，拒绝resume：{mismatches}")
    return True


def save_checkpoint(path, configuration, generation_index, next_generation, population, current_fitness_values, global_best, rng, evaluation_cache, completed_scheme_ids, cache_keys, failure_records, generation_history, all_individuals, new_openmc_runs_total):
    payload = {
        "configuration": _jsonable(configuration), "generation_index": int(generation_index),
        "next_generation": int(next_generation), "population": _jsonable(population),
        "current_population": _jsonable(population),
        "current_fitness_values": _jsonable(current_fitness_values), "global_best": _jsonable(global_best),
        "rng_state": _jsonable(rng.bit_generator.state), "evaluation_cache": _jsonable(evaluation_cache),
        "completed_scheme_ids": sorted(completed_scheme_ids), "cache_keys": sorted(cache_keys),
        "failure_records": _jsonable(failure_records), "generation_history": _jsonable(generation_history),
        "all_individuals": _jsonable(all_individuals), "new_openmc_runs_total": int(new_openmc_runs_total),
        "saved_at": datetime.now().isoformat(),
    }
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(path)
    return path


def load_checkpoint(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"找不到checkpoint：{path}")
    return json.loads(path.read_text(encoding="utf-8"))


def run_genetic_algorithm(evaluator, population_size, generations, elite_size, crossover_probability=0.8, mutation_probability=0.2, tournament_size=3, ga_seed=REAL_GA_SEED, constraints=None, checkpoint_directory=None, resume=False, allowed_position_ids=None, evaluator_name=None, runtime_counter=None):
    """运行固定代数的离散GA；代编号始终为0至generations-1。"""
    if not callable(evaluator):
        raise TypeError("evaluator必须可调用。")
    population_size, generations, elite_size = int(population_size), int(generations), int(elite_size)
    if population_size < 2 or generations < 1 or not 1 <= elite_size < population_size:
        raise ValueError("population_size、generations或elite_size不合法。")
    allowed = tuple(ALLOWED_POSITION_IDS if allowed_position_ids is None else sorted(set(int(x) for x in allowed_position_ids)))
    evaluator_name = evaluator_name or getattr(evaluator, "__name__", "evaluator")
    configuration = {
        "algorithm_version": REAL_GA_ALGORITHM_VERSION, "evaluator": evaluator_name,
        "candidate_position_count": len(allowed), "population_size": population_size,
        "generations": generations, "elite_size": elite_size,
        "crossover_probability": float(crossover_probability), "mutation_probability": float(mutation_probability),
        "tournament_size": int(tournament_size), "ga_seed": int(ga_seed),
        "geometry_model_version": REAL_GA_GEOMETRY_VERSION, "screening_profile": "screening",
        "constraints": {} if constraints is None else dict(constraints),
    }
    checkpoint_directory = Path(checkpoint_directory or REAL_GA_CHECKPOINT_ROOT / evaluator_name).resolve()
    checkpoint_directory.mkdir(parents=True, exist_ok=True)
    checkpoint_path = checkpoint_directory / "checkpoint.json"
    rng = np.random.default_rng(int(ga_seed))
    evaluation_cache, generation_history, all_individuals, failure_records = {}, [], [], []
    completed_scheme_ids, cache_keys = set(), set()
    global_best = None
    start_generation = 0
    new_openmc_runs_total = int((runtime_counter or {}).get("new_screening_openmc_runs", 0))

    if resume and checkpoint_path.is_file():
        checkpoint = load_checkpoint(checkpoint_path)
        validate_checkpoint_config(checkpoint["configuration"], configuration)
        start_generation = int(checkpoint["next_generation"])
        population = [validate_individual(item, allowed) for item in checkpoint["population"]]
        rng.bit_generator.state = checkpoint["rng_state"]
        evaluation_cache = dict(checkpoint.get("evaluation_cache", {}))
        generation_history = list(checkpoint.get("generation_history", []))
        all_individuals = list(checkpoint.get("all_individuals", []))
        failure_records = list(checkpoint.get("failure_records", []))
        completed_scheme_ids = set(checkpoint.get("completed_scheme_ids", []))
        cache_keys = set(checkpoint.get("cache_keys", []))
        global_best = checkpoint.get("global_best")
        new_openmc_runs_total = int(checkpoint.get("new_openmc_runs_total", new_openmc_runs_total))
        if runtime_counter is not None:
            runtime_counter["new_screening_openmc_runs"] = new_openmc_runs_total
    else:
        population = initialize_population(population_size, REFERENCE_DUMMY_IDS, rng, allowed)

    for generation in range(start_generation, generations):
        generation_start = time.perf_counter()
        fitness_records = {}
        new_runs_before = int((runtime_counter or {}).get("new_screening_openmc_runs", 0))
        generation_cache_hits = 0
        generation_failures = 0
        for individual_index, individual in enumerate(population):
            individual = validate_individual(individual, allowed)
            memory_key = ",".join(str(value) for value in individual)
            if memory_key in evaluation_cache:
                result = dict(evaluation_cache[memory_key])
                result["cache_hit"] = True
                result["calculation_seconds"] = 0.0
            else:
                result = dict(evaluator(individual))
                evaluation_cache[memory_key] = dict(result)
            if result.get("cache_hit"):
                generation_cache_hits += 1
            fitness = compute_fitness(result, constraints)
            record = {**result, **fitness}
            fitness_records[individual] = record
            scheme_id = str(result.get("scheme_id", scheme_key(individual)))
            if result.get("status") in {"completed", "mock_completed"}:
                completed_scheme_ids.add(scheme_id)
            else:
                generation_failures += 1
                failure_records.append({"generation": generation, "scheme_id": scheme_id, "status": result.get("status"), "error_message": result.get("error_message")})
            if result.get("cache_key"):
                cache_keys.add(str(result["cache_key"]))
            all_individuals.append({
                "generation": generation, "individual_index": individual_index, "scheme_id": scheme_id,
                "dummy_ids": list(individual), "fitness": float(fitness["fitness"]),
                "Fr": result.get("radial_peaking_factor"), "CV": result.get("power_coefficient_of_variation"),
                "keff": result.get("keff"), "keff_std": result.get("keff_std"),
                "rho": result.get("reactivity", result.get("rho")), "rho_mk": result.get("rho_mk"),
                "max_power_position_id": result.get("max_power_position_id"),
                "mean_pin_relative_uncertainty": result.get("mean_pin_relative_uncertainty"),
                "cache_hit": bool(result.get("cache_hit", False)), "status": result.get("status"),
                "calculation_seconds": float(result.get("calculation_seconds", 0.0)),
            })

        ranked = sorted(population, key=lambda ind: float(fitness_records[ind]["fitness"]))
        generation_best = ranked[0]
        best_record = fitness_records[generation_best]
        if global_best is None or float(best_record["fitness"]) < float(global_best["fitness"]):
            global_best = {"scheme": list(generation_best), **_jsonable(best_record)}
        fitness_values = np.asarray([fitness_records[ind]["fitness"] for ind in population], dtype=float)
        new_runs_after = int((runtime_counter or {}).get("new_screening_openmc_runs", new_runs_before))
        generation_history.append({
            "generation": generation, "population_size": population_size,
            "best_fitness": float(np.min(fitness_values)), "mean_fitness": float(np.mean(fitness_values)),
            "median_fitness": float(np.median(fitness_values)), "worst_fitness": float(np.max(fitness_values)),
            "global_best_fitness": float(global_best["fitness"]),
            "best_scheme": list(generation_best), "global_best_scheme": list(global_best["scheme"]),
            "best_keff": best_record.get("keff"), "best_keff_std": best_record.get("keff_std"),
            "best_rho_mk": best_record.get("rho_mk"), "best_Fr": best_record.get("radial_peaking_factor"),
            "best_CV": best_record.get("power_coefficient_of_variation"),
            "new_openmc_runs": new_runs_after - new_runs_before, "cache_hits": generation_cache_hits,
            "failures": generation_failures, "generation_wall_time": time.perf_counter() - generation_start,
        })
        next_population = (
            create_next_generation(population, fitness_records, population_size, elite_size, crossover_probability, mutation_probability, tournament_size, rng, allowed)
            if generation + 1 < generations else list(population)
        )
        current_values = {",".join(map(str, ind)): float(fitness_records[ind]["fitness"]) for ind in population}
        save_checkpoint(
            checkpoint_path, configuration, generation, generation + 1, next_population, current_values,
            global_best, rng, evaluation_cache, completed_scheme_ids, cache_keys, failure_records,
            generation_history, all_individuals, new_runs_after,
        )
        pd.DataFrame(generation_history).to_csv(checkpoint_directory / "generation_history.csv", index=False, encoding="utf-8")
        pd.DataFrame(all_individuals).to_csv(checkpoint_directory / "all_individuals.csv", index=False, encoding="utf-8")
        population = next_population

    return {
        "configuration": configuration, "generation_history": generation_history,
        "all_individuals": all_individuals, "evaluation_cache": evaluation_cache,
        "global_best": global_best, "checkpoint_file": str(checkpoint_path),
        "new_openmc_runs_total": int((runtime_counter or {}).get("new_screening_openmc_runs", new_openmc_runs_total)),
        "failure_records": failure_records,
    }


## 10 真实OpenMC screening评价接口

完整cache identity包含方案、几何/材料/计数器版本、profile、粒子数、批次、非激活批次和方案绑定OpenMC seed。参考排布screening严格复用已有 `iaea_reference_v1` 结果；其他方案写入独立目录。screening与final都只建立345棒 `kappa-fission` tally和10×10×10 Shannon熵网格，不增加能谱或XY通量网格。


In [ ]:
class RealGASafetyLimitError(RuntimeError):
    pass


REAL_GA_RUNTIME_COUNTER = {"new_screening_openmc_runs": 0, "new_final_openmc_runs": 0}


def _sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def get_openmc_seed(normalized_scheme, profile):
    scheme = validate_individual(normalized_scheme)
    if profile not in RUN_PROFILES:
        raise ValueError(f"未知profile：{profile}")
    if scheme == tuple(REFERENCE_DUMMY_IDS) and profile == "screening":
        return OPENMC_BASE_SEED
    text = f"{OPENMC_BASE_SEED}|{profile}|" + ",".join(map(str, scheme))
    return int(hashlib.sha256(text.encode("utf-8")).hexdigest(), 16) % 2_147_483_646 + 1


def _real_ga_cache_identity(scheme, profile):
    scheme = validate_individual(scheme)
    profile_values = RUN_PROFILES[profile]
    try:
        cross_sections = Path(openmc.config["cross_sections"]).expanduser().resolve()
    except KeyError as error:
        raise FileNotFoundError("尚未配置OpenMC截面库。") from error
    if not cross_sections.is_file():
        raise FileNotFoundError(f"OpenMC截面库不存在：{cross_sections}")
    identity = {
        "normalized_dummy_ids": list(scheme), "scheme_id": scheme_key(scheme),
        "geometry_model_version": REAL_GA_GEOMETRY_VERSION, "model_variant": REAL_GA_MODEL_VARIANT,
        "control_rod_state": REAL_GA_CONTROL_ROD_STATE, "profile_name": profile,
        "particles": int(profile_values["particles"]), "batches": int(profile_values["batches"]),
        "inactive": int(profile_values["inactive"]), "openmc_seed": get_openmc_seed(scheme, profile),
        "tally_version": REAL_GA_TALLY_VERSION, "material_model_version": REAL_GA_MATERIAL_VERSION,
        "materials_notebook_sha256": _sha256_file(materials_notebook_path),
        "geometry_notebook_sha256": _sha256_file(geometry_notebook_path),
        "cross_sections": str(cross_sections), "cross_sections_sha256": _sha256_file(cross_sections),
        "openmc_version": openmc.__version__,
    }
    cache_key = hashlib.sha256(json.dumps(identity, sort_keys=True, separators=(",", ":")).encode("utf-8")).hexdigest()
    return identity, cache_key


def _find_real_openmc_executable():
    candidates = [Path(sys.executable).with_name("openmc"), Path(sys.prefix) / "envs" / "openmc-env" / "bin" / "openmc"]
    executable = next((candidate.resolve() for candidate in candidates if candidate.is_file()), None)
    if executable is None:
        raise FileNotFoundError("找不到OpenMC可执行程序。")
    return executable


def _prepare_real_ga_materials(scheme_geometry, cross_sections_path, expected_beryllium_cells):
    root = ET.parse(cross_sections_path).getroot()
    thermal_names = {library.get("materials") for library in root.findall("library") if library.get("type") == "thermal"}
    original = material_map["beryllium"]
    be_cells = [cell for cell in scheme_geometry.get_all_cells().values() if cell.fill is original]
    if len(be_cells) != int(expected_beryllium_cells):
        raise RuntimeError(f"铍反射层Cell数量错误：预期{expected_beryllium_cells}，实际{len(be_cells)}。")
    if "c_Be_in_Be" in thermal_names:
        return materials, {}
    if "c_Be" not in thermal_names:
        raise FileNotFoundError("截面库不含c_Be_in_Be或兼容的c_Be。")
    replacement = openmc.Material(name=original.name)
    for nuclide in original.nuclides:
        replacement.add_nuclide(nuclide.name, nuclide.percent, nuclide.percent_type)
    replacement.set_density(original.density_units, original.density)
    if original.temperature is not None:
        replacement.temperature = cast(Real, original.temperature)
    replacement.add_s_alpha_beta("c_Be")
    for cell in be_cells:
        cell.fill = replacement
    return openmc.Materials([replacement if material is original else material for material in materials]), {"c_Be_in_Be": "c_Be"}


def _entropy_tail_summary(entropy_values, inactive):
    active = np.asarray(entropy_values, dtype=float)[int(inactive):]
    tail = active[-min(10, active.size):]
    x = np.arange(tail.size, dtype=float)
    slope = float(np.polyfit(x, tail, 1)[0]) if tail.size >= 2 else 0.0
    return {
        "tail_batch_count": int(tail.size), "first_value": float(tail[0]), "last_value": float(tail[-1]),
        "absolute_first_to_last_change": float(abs(tail[-1] - tail[0])), "range": float(np.ptp(tail)),
        "linear_slope_per_batch": slope, "heuristically_stable": bool(np.ptp(tail) <= 0.10 and abs(slope) <= 0.005),
    }


def _reuse_reference_screening(identity, cache_key, run_directory):
    reference_directory = Path("build/results/reference/iaea_reference_v1/screening").resolve()
    result_path = reference_directory / "result.json"
    metadata_path = reference_directory / "run_metadata.json"
    if not result_path.is_file() or not metadata_path.is_file():
        return None
    old = json.loads(result_path.read_text(encoding="utf-8"))
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    expected = {
        "geometry_model_version": REAL_GA_GEOMETRY_VERSION, "model_variant": REAL_GA_MODEL_VARIANT,
        "control_rod_state": REAL_GA_CONTROL_ROD_STATE, "particles": identity["particles"],
        "batches": identity["batches"], "inactive": identity["inactive"], "random_seed": identity["openmc_seed"],
    }
    if any(metadata.get(key) != value for key, value in expected.items()):
        raise ValueError("已有参考screening metadata与真实GA cache identity不一致。")
    if tuple(old.get("reference_dummy_position_ids", [])) != tuple(REFERENCE_DUMMY_IDS) or old.get("status") != "completed":
        raise ValueError("已有参考screening结果方案或状态不正确。")
    for key in ("pin_power_file", "generation_history_file", "entropy_history_file", "statepoint_file", "openmc_output_file"):
        if not Path(old[key]).is_file():
            raise FileNotFoundError(f"已有参考screening缓存缺少文件：{old[key]}")
    entropy = pd.read_csv(old["entropy_history_file"])["shannon_entropy"].to_numpy(float)
    result = {
        "cache_identity": identity, "cache_key": cache_key, "scheme_id": identity["scheme_id"],
        "normalized_dummy_ids": list(REFERENCE_DUMMY_IDS), "geometry_model_version": REAL_GA_GEOMETRY_VERSION,
        "model_variant": REAL_GA_MODEL_VARIANT, "control_rod_state": REAL_GA_CONTROL_ROD_STATE,
        "profile": "screening", "openmc_seed": identity["openmc_seed"], "particles": identity["particles"],
        "batches": identity["batches"], "inactive": identity["inactive"], "keff": float(old["keff_mean"]),
        "keff_std": float(old["keff_std"]), "reactivity": float(old["reactivity"]),
        "rho_mk": float(old["reactivity"] * 1000.0), "radial_peaking_factor": float(old["radial_peaking_factor"]),
        "power_coefficient_of_variation": float(old["power_coefficient_of_variation"]),
        "max_power_position_id": int(old["max_power_position_id"]),
        "mean_pin_relative_uncertainty": float(old["mean_pin_relative_uncertainty"]),
        "median_pin_relative_uncertainty": float(old["median_pin_relative_uncertainty"]),
        "p95_pin_relative_uncertainty": float(old["p95_pin_relative_uncertainty"]),
        "max_pin_relative_uncertainty": float(old["max_pin_relative_uncertainty"]),
        "entropy_tail_summary": _entropy_tail_summary(entropy, identity["inactive"]),
        "calculation_seconds": 0.0, "cache_hit": True, "cache_source": "existing_reference_screening",
        "run_directory": str(reference_directory), "statepoint_file": old["statepoint_file"],
        "pin_power_file": old["pin_power_file"], "generation_history_file": old["generation_history_file"],
        "entropy_history_file": old["entropy_history_file"], "openmc_output_file": old["openmc_output_file"],
        "status": "completed",
    }
    run_directory.mkdir(parents=True, exist_ok=True)
    _write_json(run_directory / "scheme.json", identity)
    _write_json(run_directory / "result.json", result)
    return result


def evaluate_scheme_real(dummy_position_ids, profile_name="screening", force_rerun=False, run_openmc=True, allow_real_openmc=None, runtime_counter=None, max_new_screening_cases=MAX_NEW_SCREENING_OPENMC_CASES, max_new_final_cases=MAX_NEW_FINAL_OPENMC_CASES):
    """使用完整身份缓存评价真实方案；screening与final均受独立硬上限保护。"""
    scheme = validate_individual(dummy_position_ids)
    if profile_name not in {"debug", "screening", "final"}:
        raise ValueError("profile_name只允许debug、screening或final。")
    allow = ALLOW_REAL_OPENMC if allow_real_openmc is None else bool(allow_real_openmc)
    runtime_counter = REAL_GA_RUNTIME_COUNTER if runtime_counter is None else runtime_counter
    identity, cache_key = _real_ga_cache_identity(scheme, profile_name)
    run_root = (REAL_GA_ROOT / "finalists") if profile_name == "final" else REAL_GA_SCHEMES_ROOT
    run_directory = run_root / identity["scheme_id"] / profile_name / cache_key
    result_path, error_path = run_directory / "result.json", run_directory / "error.json"
    if result_path.is_file() and not force_rerun:
        cached = json.loads(result_path.read_text(encoding="utf-8"))
        if cached.get("cache_identity") == identity and cached.get("cache_key") == cache_key and cached.get("status") == "completed":
            cached["cache_hit"] = True
            cached["calculation_seconds"] = 0.0
            return cached
        if cached.get("cache_identity") == identity and cached.get("cache_key") == cache_key and cached.get("status") == "failed":
            force_rerun = True
        else:
            raise ValueError("同目录旧结果的完整cache identity不匹配，拒绝复用。")
    if scheme == tuple(REFERENCE_DUMMY_IDS) and profile_name == "screening" and not force_rerun:
        reused = _reuse_reference_screening(identity, cache_key, run_directory)
        if reused is not None:
            return reused
    if run_openmc and profile_name in {"screening", "final"} and not allow:
        raise PermissionError("ALLOW_REAL_OPENMC=False，禁止真实OpenMC运行。")

    run_directory.mkdir(parents=True, exist_ok=True)
    _write_json(run_directory / "scheme.json", identity)
    start = time.perf_counter()
    try:
        geometry, metadata = build_geometry_for_scheme(
            scheme, control_rod_state=REAL_GA_CONTROL_ROD_STATE, export_directory=None,
            create_plots=False, model_variant=REAL_GA_MODEL_VARIANT, top_be_thickness_cm=0.0,
        )
        if metadata.get("geometry_model_version") != REAL_GA_GEOMETRY_VERSION or metadata.get("model_variant") != REAL_GA_MODEL_VARIANT:
            raise RuntimeError("候选几何版本或变体错误。")
        expected_counts = {"fuel": 345, "dummy": 5, "tie_rod": 4, "central": 1}
        if any(int(metadata["counts"][key]) != value for key, value in expected_counts.items()):
            raise RuntimeError(f"候选几何构件数量错误：{metadata['counts']}")
        materials_for_run, aliases = _prepare_real_ga_materials(
            geometry, Path(identity["cross_sections"]), 2 + int(float(metadata.get("top_be_thickness_cm", 0.0)) > 0.0),
        )
        materials_for_run.cross_sections = identity["cross_sections"]
        settings = create_settings(profile_name, identity["openmc_seed"])
        entropy_mesh = openmc.RegularMesh(name="Real GA entropy mesh")
        entropy_mesh.lower_left = (-11.5, -11.5, -11.5)
        entropy_mesh.upper_right = (11.5, 11.5, 11.5)
        entropy_mesh.dimension = (10, 10, 10)
        settings.entropy_mesh = entropy_mesh
        tallies, fuel_ids = create_power_tally(metadata["fuel_meat_cells"])
        if len(tallies) != 1 or tallies[0].name != "fuel_pin_kappa_fission":
            raise RuntimeError("真实screening只能包含345棒kappa-fission Tally。")
        model = openmc.Model(materials=materials_for_run, geometry=geometry, settings=settings, tallies=tallies)
        model.export_to_xml(directory=run_directory)
        if not run_openmc:
            exported = {"cache_identity": identity, "cache_key": cache_key, "scheme_id": identity["scheme_id"], "normalized_dummy_ids": list(scheme), "profile": profile_name, "run_directory": str(run_directory), "cache_hit": False, "calculation_seconds": time.perf_counter() - start, "status": "exported"}
            _write_json(result_path, exported)
            return exported
        if profile_name == "screening" and int(runtime_counter.get("new_screening_openmc_runs", 0)) >= int(max_new_screening_cases):
            raise RealGASafetyLimitError(f"新的screening OpenMC调用将超过硬上限{max_new_screening_cases}。")
        if profile_name == "final" and int(runtime_counter.get("new_final_openmc_runs", 0)) >= int(max_new_final_cases):
            raise RealGASafetyLimitError(f"新的final OpenMC调用将超过硬上限{max_new_final_cases}。")
        counter_key = "new_screening_openmc_runs" if profile_name == "screening" else "new_final_openmc_runs"
        runtime_counter[counter_key] = int(runtime_counter.get(counter_key, 0)) + 1
        output_buffer = io.StringIO()
        from unittest.mock import PropertyMock, patch
        with (
            contextlib.redirect_stdout(output_buffer), contextlib.redirect_stderr(output_buffer),
            patch.object(openmc.Model, "is_initialized", new_callable=PropertyMock, return_value=False),
        ):
            returned = model.run(cwd=run_directory, openmc_exec=_find_real_openmc_executable(), export_model_xml=False)
        output_path = run_directory / "openmc_output.txt"
        output_path.write_text(output_buffer.getvalue(), encoding="utf-8")
        statepoint_path = Path(returned) if returned is not None else sorted(run_directory.glob("statepoint.*.h5"))[-1]
        if not statepoint_path.is_absolute():
            statepoint_path = run_directory / statepoint_path
        with openmc.StatePoint(statepoint_path) as statepoint:
            keff = statepoint.keff
            keff_mean, keff_std = float(keff.nominal_value), float(keff.std_dev)
            generation_values = np.asarray(statepoint.k_generation, dtype=float).reshape(-1)
            try:
                entropy_values = np.asarray(statepoint.entropy, dtype=float).reshape(-1)
            except (AttributeError, KeyError):
                entropy_values = None
            tally = statepoint.get_tally(name="fuel_pin_kappa_fission")
            pin_mean = np.asarray(tally.mean, dtype=float).reshape(-1)
            pin_std = np.asarray(tally.std_dev, dtype=float).reshape(-1)
        if entropy_values is None:
            with h5py.File(statepoint_path, "r") as statepoint_h5:
                entropy_values = np.asarray(statepoint_h5["entropy"][...], dtype=float).reshape(-1)
        if generation_values.size != identity["batches"] or entropy_values.size != identity["batches"]:
            raise ValueError("逐批keff或Shannon熵长度不符合计算批次。")
        if pin_mean.size != 345 or pin_std.size != 345 or np.any(pin_mean <= 0.0):
            raise ValueError("燃料棒Tally不是345项或含非正均值。")
        if not all(np.all(np.isfinite(values)) for values in (generation_values, entropy_values, pin_mean, pin_std)):
            raise ValueError("真实screening statepoint含非有限数据。")
        relative_power = pin_mean / float(np.mean(pin_mean))
        relative_uncertainty = pin_std / pin_mean
        if not math.isclose(float(np.mean(relative_power)), 1.0, rel_tol=0.0, abs_tol=1.0e-12):
            raise ValueError("345棒relative_power平均值不是1。")
        positions = ordinary_positions_df[["ordinary_position_id", "ring_index", "local_index", "x_cm", "y_cm"]]
        pin_table = pd.DataFrame({
            "ordinary_position_id": fuel_ids, "kappa_fission_mean": pin_mean, "kappa_fission_std": pin_std,
            "relative_power": relative_power, "relative_uncertainty": relative_uncertainty,
        }).merge(positions, on="ordinary_position_id", how="left", validate="one_to_one")
        pin_path = run_directory / "pin_power.csv"
        pin_table.to_csv(pin_path, index=False, encoding="utf-8")
        phase = np.where(np.arange(1, identity["batches"] + 1) <= identity["inactive"], "inactive", "active")
        generation_path = run_directory / "generation_history.csv"
        entropy_path = run_directory / "entropy_history.csv"
        pd.DataFrame({"batch": np.arange(1, identity["batches"] + 1), "phase": phase, "k_generation": generation_values}).to_csv(generation_path, index=False, encoding="utf-8")
        pd.DataFrame({"batch": np.arange(1, identity["batches"] + 1), "phase": phase, "shannon_entropy": entropy_values}).to_csv(entropy_path, index=False, encoding="utf-8")
        rho = (keff_mean - 1.0) / keff_mean
        max_index = int(np.argmax(relative_power))
        result = {
            "cache_identity": identity, "cache_key": cache_key, "scheme_id": identity["scheme_id"],
            "normalized_dummy_ids": list(scheme), "geometry_model_version": REAL_GA_GEOMETRY_VERSION,
            "model_variant": REAL_GA_MODEL_VARIANT, "control_rod_state": REAL_GA_CONTROL_ROD_STATE,
            "profile": profile_name, "openmc_seed": identity["openmc_seed"], "particles": identity["particles"],
            "batches": identity["batches"], "inactive": identity["inactive"], "keff": keff_mean, "keff_std": keff_std,
            "reactivity": float(rho), "rho_mk": float(rho * 1000.0),
            "radial_peaking_factor": float(np.max(relative_power)),
            "power_coefficient_of_variation": float(np.std(relative_power, ddof=0)),
            "max_power_position_id": int(fuel_ids[max_index]),
            "mean_pin_relative_uncertainty": float(np.mean(relative_uncertainty)),
            "median_pin_relative_uncertainty": float(np.median(relative_uncertainty)),
            "p95_pin_relative_uncertainty": float(np.percentile(relative_uncertainty, 95)),
            "max_pin_relative_uncertainty": float(np.max(relative_uncertainty)),
            "entropy_tail_summary": _entropy_tail_summary(entropy_values, identity["inactive"]),
            "calculation_seconds": float(time.perf_counter() - start), "cache_hit": False,
            "thermal_scattering_aliases": aliases, "run_directory": str(run_directory),
            "statepoint_file": str(statepoint_path.resolve()), "pin_power_file": str(pin_path.resolve()),
            "generation_history_file": str(generation_path.resolve()), "entropy_history_file": str(entropy_path.resolve()),
            "openmc_output_file": str(output_path.resolve()), "status": "completed",
        }
        _write_json(result_path, result)
        error_path.unlink(missing_ok=True)
        return result
    except RealGASafetyLimitError:
        raise
    except Exception as error:
        failed = {
            "cache_identity": identity, "cache_key": cache_key, "scheme_id": identity["scheme_id"],
            "normalized_dummy_ids": list(scheme), "profile": profile_name, "calculation_seconds": float(time.perf_counter() - start),
            "cache_hit": False, "status": "failed", "error_type": type(error).__name__, "error_message": str(error),
        }
        _write_json(error_path, failed)
        _write_json(result_path, failed)
        return failed


## 11 纯算法验证与mock回归

本节不调用OpenMC。测试严格输入校验、repair、初始化、100次交叉、100次强制变异、确定性重放、15代mock回归、精英保留、缓存和checkpoint恢复。旧mock源码已不存在，因此不伪造逐点一致的旧轨迹。


In [11]:
algorithm_test_results = {}

valid = validate_individual((0, 50, 100, 200, 349))
algorithm_test_results["valid_individual"] = valid == (0, 50, 100, 200, 349)
invalid_cases = {
    "duplicate": (0, 0, 1, 2, 3), "out_of_range": (0, 1, 2, 3, 350),
    "non_integer": (0, 1, 2, 3, 4.5), "too_short": (0, 1, 2, 3),
    "too_long": (0, 1, 2, 3, 4, 5),
}
for name, candidate in invalid_cases.items():
    try:
        validate_individual(candidate)
    except ValueError:
        algorithm_test_results[f"reject_{name}"] = True
    else:
        raise AssertionError(f"严格校验没有拒绝{name}。")

test_rng = np.random.default_rng(REAL_GA_SEED)
for _ in range(100):
    repaired = repair_individual([0, 0, -1, 350, int(test_rng.integers(0, 350))], test_rng)
    validate_individual(repaired)
algorithm_test_results["repair_100"] = True

population = initialize_population(20, REFERENCE_DUMMY_IDS, np.random.default_rng(REAL_GA_SEED))
assert population[0] == tuple(REFERENCE_DUMMY_IDS) and len(population) == len(set(population)) == 20
assert all(validate_individual(individual) == individual for individual in population)
algorithm_test_results["initialize_population_20"] = True

crossover_rng = np.random.default_rng(REAL_GA_SEED + 1)
for _ in range(100):
    parent_indices = crossover_rng.choice(len(population), size=2, replace=False)
    children = set_crossover(population[int(parent_indices[0])], population[int(parent_indices[1])], crossover_rng)
    assert all(validate_individual(child) == child for child in children)
algorithm_test_results["crossover_100"] = True

mutation_rng = np.random.default_rng(REAL_GA_SEED + 2)
mutation_changes = 0
for _ in range(100):
    original = population[int(mutation_rng.integers(0, len(population)))]
    mutated = replace_one_mutation(original, 1.0, mutation_rng)
    validate_individual(mutated)
    mutation_changes += int(mutated != original)
assert mutation_changes == 100
algorithm_test_results["mutation_100_changed"] = True

replay_a = initialize_population(20, REFERENCE_DUMMY_IDS, np.random.default_rng(REAL_GA_SEED))
replay_b = initialize_population(20, REFERENCE_DUMMY_IDS, np.random.default_rng(REAL_GA_SEED))
assert replay_a == replay_b
algorithm_test_results["deterministic_replay"] = True

mock_checkpoint_directory = REAL_GA_TEST_ROOT / "mock_regression_v2"
mock_constraints = {"rho_min": None, "rho_max": None, "penalty_weight": 10000.0, "failure_penalty": FAILURE_PENALTY}
mock_run = run_genetic_algorithm(
    mock_evaluate_scheme, population_size=20, generations=15, elite_size=2,
    crossover_probability=0.8, mutation_probability=0.2, tournament_size=3,
    ga_seed=REAL_GA_SEED, constraints=mock_constraints,
    checkpoint_directory=mock_checkpoint_directory, resume=False, evaluator_name="mock_regression_v2",
)
assert len(mock_run["generation_history"]) == 15
best_series = np.asarray([row["best_fitness"] for row in mock_run["generation_history"]], dtype=float)
assert np.all(np.diff(best_series) <= 1.0e-15)
mock_resumed = run_genetic_algorithm(
    mock_evaluate_scheme, population_size=20, generations=15, elite_size=2,
    crossover_probability=0.8, mutation_probability=0.2, tournament_size=3,
    ga_seed=REAL_GA_SEED, constraints=mock_constraints,
    checkpoint_directory=mock_checkpoint_directory, resume=True, evaluator_name="mock_regression_v2",
)
assert mock_resumed["generation_history"] == mock_run["generation_history"]
assert len(mock_run["evaluation_cache"]) < 20 * 15
algorithm_test_results["mock_15_generations"] = True
algorithm_test_results["elite_retention"] = True
algorithm_test_results["checkpoint_resume"] = True
algorithm_test_results["memory_cache"] = True

seed_reference = get_openmc_seed(REFERENCE_DUMMY_IDS, "screening")
assert seed_reference == 20260730
seed_other_a = get_openmc_seed((0, 50, 100, 200, 349), "screening")
seed_other_b = get_openmc_seed((0, 50, 100, 200, 349), "screening")
assert seed_other_a == seed_other_b and seed_other_a != seed_reference and seed_other_a > 0
algorithm_test_results["stable_openmc_seed"] = True

identity_debug, key_debug = _real_ga_cache_identity(REFERENCE_DUMMY_IDS, "debug")
identity_screening, key_screening = _real_ga_cache_identity(REFERENCE_DUMMY_IDS, "screening")
assert key_debug != key_screening and identity_debug["profile_name"] != identity_screening["profile_name"]
algorithm_test_results["profile_cache_isolation"] = True

ALGORITHM_VALIDATION_PATH = REAL_GA_TEST_ROOT / "algorithm_validation.json"
_write_json(ALGORITHM_VALIDATION_PATH, {
    "status": "passed", "tests": algorithm_test_results,
    "mock_regression_status": "passed_without_claiming_old_trajectory_reproduction",
    "mock_generations": 15, "mock_population": 20,
    "old_mock_source_available": False,
    "note": "旧mock源码不在当前Notebook中，因此只验证论文算法逻辑、确定性、缓存和checkpoint。",
})
print("纯算法验证全部通过：", len(algorithm_test_results), "项")
display(pd.Series(algorithm_test_results, name="algorithm tests"))


纯算法验证全部通过： 17 项


valid_individual            True
reject_duplicate            True
reject_out_of_range         True
reject_non_integer          True
reject_too_short            True
reject_too_long             True
repair_100                  True
initialize_population_20    True
crossover_100               True
mutation_100_changed        True
deterministic_replay        True
mock_15_generations         True
elite_retention             True
checkpoint_resume           True
memory_cache                True
stable_openmc_seed          True
profile_cache_isolation     True
Name: algorithm tests, dtype: bool

## 12 真实OpenMC screening pilot

pilot固定为6个体、2代、2个精英，最多10次新的screening OpenMC调用。Notebook默认不运行；只有同时临时设置 `ALLOW_REAL_OPENMC=True` 和 `RUN_REAL_GA_PILOT=True` 才执行。参考排布优先命中已有screening缓存。


In [13]:
PILOT_CONFIGURATION = {
    "population_size": 6, "generations": 2, "elite_size": 2,
    "crossover_probability": 0.8, "mutation_probability": 0.2,
    "tournament_size": 3, "ga_seed": REAL_GA_SEED, "patience": None,
    "screening_profile": "screening", "particles": 5000, "batches": 60,
    "inactive": 15, "rho_min": None, "rho_max": None,
    "max_new_screening_openmc_cases": MAX_NEW_SCREENING_OPENMC_CASES,
}
PILOT_CONFIG_HASH = hashlib.sha256(
    json.dumps(PILOT_CONFIGURATION, sort_keys=True, separators=(",", ":")).encode("utf-8")
).hexdigest()
PILOT_CHECKPOINT_DIRECTORY = REAL_GA_CHECKPOINT_ROOT / f"pilot_{PILOT_CONFIG_HASH[:12]}"

print("REAL GA PILOT RUN CONFIRMATION")
print("geometry model:", REAL_GA_GEOMETRY_VERSION, REAL_GA_MODEL_VARIANT)
print("control rod state:", REAL_GA_CONTROL_ROD_STATE)
print("candidate positions:", len(ALLOWED_POSITION_IDS))
print("reference dummy IDs:", tuple(REFERENCE_DUMMY_IDS))
print("population / generations / generation range:", 6, 2, "0..1")
print("elite / crossover / mutation / tournament:", 2, 0.8, 0.2, 3)
print("GA seed:", REAL_GA_SEED)
print("screening particles / batches / inactive:", 5000, 60, 15)
print("reference screening seed:", get_openmc_seed(REFERENCE_DUMMY_IDS, "screening"))
print("rho_min / rho_max / reaction penalty:", None, None, False)
print("global safety OpenMC case cap:", MAX_NEW_SCREENING_OPENMC_CASES)
print("output directory:", REAL_GA_PILOT_ROOT)
print("mock evaluator = OFF")
print("real OpenMC evaluator =", "ON" if ALLOW_REAL_OPENMC and RUN_REAL_GA_PILOT else "OFF")

pilot_run = None
if RUN_REAL_GA_PILOT:
    if not ALLOW_REAL_OPENMC:
        raise PermissionError("RUN_REAL_GA_PILOT=True但ALLOW_REAL_OPENMC=False。")
    REAL_GA_RUNTIME_COUNTER["new_screening_openmc_runs"] = 0
    pilot_manifest = {
        **PILOT_CONFIGURATION, "config_hash": PILOT_CONFIG_HASH,
        "geometry_model_version": REAL_GA_GEOMETRY_VERSION, "model_variant": REAL_GA_MODEL_VARIANT,
        "control_rod_state": REAL_GA_CONTROL_ROD_STATE, "reference_dummy_ids": list(REFERENCE_DUMMY_IDS),
        "candidate_position_count": len(ALLOWED_POSITION_IDS), "openmc_version": openmc.__version__,
        "start_time": datetime.now().isoformat(), "allow_real_openmc": True,
    }
    _write_json(REAL_GA_PILOT_ROOT / "pilot_run_manifest.json", pilot_manifest)

    def pilot_real_evaluator(individual):
        return evaluate_scheme_real(
            individual, profile_name="screening", force_rerun=False, run_openmc=True,
            allow_real_openmc=True, runtime_counter=REAL_GA_RUNTIME_COUNTER,
            max_new_screening_cases=MAX_NEW_SCREENING_OPENMC_CASES,
        )

    pilot_start = time.perf_counter()
    pilot_run = run_genetic_algorithm(
        pilot_real_evaluator, population_size=6, generations=2, elite_size=2,
        crossover_probability=0.8, mutation_probability=0.2, tournament_size=3,
        ga_seed=REAL_GA_SEED,
        constraints={"rho_min": None, "rho_max": None, "penalty_weight": 10000.0, "failure_penalty": FAILURE_PENALTY},
        checkpoint_directory=PILOT_CHECKPOINT_DIRECTORY, resume=True,
        evaluator_name="real_openmc_screening_pilot", runtime_counter=REAL_GA_RUNTIME_COUNTER,
    )
    pilot_wall_time = time.perf_counter() - pilot_start
    pd.DataFrame(pilot_run["generation_history"]).to_csv(
        REAL_GA_PILOT_ROOT / "pilot_generation_history.csv", index=False, encoding="utf-8"
    )
    pd.DataFrame(pilot_run["all_individuals"]).to_csv(
        REAL_GA_PILOT_ROOT / "pilot_all_individuals.csv", index=False, encoding="utf-8"
    )
    unique_rows = list(pilot_run["evaluation_cache"].values())
    pd.DataFrame(unique_rows).to_csv(
        REAL_GA_PILOT_ROOT / "pilot_unique_evaluations.csv", index=False, encoding="utf-8"
    )
    successful = [row for row in unique_rows if row.get("status") == "completed"]
    if len(successful) != len(unique_rows):
        raise RuntimeError("pilot出现失败方案，停止后续就绪判定。")
    for result in successful:
        if not (math.isfinite(float(result["keff"])) and math.isfinite(float(result["keff_std"]))):
            raise ValueError("pilot keff含非有限值。")
        if not (math.isfinite(float(result["radial_peaking_factor"])) and float(result["radial_peaking_factor"]) > 1.0):
            raise ValueError("pilot Fr不是大于1的有限值。")
        if not math.isclose(compute_fitness(result, {"rho_min": None, "rho_max": None})["fitness"], float(result["radial_peaking_factor"]), rel_tol=0.0, abs_tol=1e-14):
            raise ValueError("真实pilot成功方案fitness不等于Fr。")
    best_scheme = tuple(pilot_run["global_best"]["scheme"])
    runs_before_cache_check = REAL_GA_RUNTIME_COUNTER["new_screening_openmc_runs"]
    repeated = pilot_real_evaluator(best_scheme)
    assert repeated["cache_hit"] is True
    assert REAL_GA_RUNTIME_COUNTER["new_screening_openmc_runs"] == runs_before_cache_check
    print("真实screening pilot完成，新OpenMC调用：", runs_before_cache_check)
else:
    print("安全默认：未运行真实OpenMC pilot。")


REAL GA PILOT RUN CONFIRMATION
geometry model: iaea_reference_v1 iaea_reference
control rod state: withdrawn
candidate positions: 350
reference dummy IDs: (285, 298, 311, 324, 337)
population / generations / generation range: 6 2 0..1
elite / crossover / mutation / tournament: 2 0.8 0.2 3
GA seed: 20260728
screening particles / batches / inactive: 5000 60 15
reference screening seed: 20260730
rho_min / rho_max / reaction penalty: None None False
global safety OpenMC case cap: 10
output directory: ~/SZU_MNSR_Opt/build/optimization/real_ga/pilot
mock evaluator = OFF
real OpenMC evaluator = ON


真实screening pilot完成，新OpenMC调用： 9


## 13 pilot统计与正式GA规模建议

只在pilot完成后生成验证汇总。正式规模比较采用精英不重复计算时的最大唯一方案数估计：`population + (generations-1) × (population-elite)`；预计用时基于pilot新算例中位耗时，不启动任何推荐规模搜索。


In [14]:
GA_PILOT_VALIDATION_PATH = REAL_GA_PILOT_ROOT / "ga_pilot_validation.json"
if pilot_run is not None:
    all_records = pd.DataFrame(pilot_run["all_individuals"])
    unique_results = list(pilot_run["evaluation_cache"].values())
    successful_results = [result for result in unique_results if result.get("status") == "completed"]
    new_results = [result for result in successful_results if not result.get("cache_hit") and float(result.get("calculation_seconds", 0.0)) > 0.0]
    times = np.asarray([result["calculation_seconds"] for result in new_results], dtype=float)
    uncertainties = np.asarray([result["mean_pin_relative_uncertainty"] for result in successful_results], dtype=float)
    fr_values = np.asarray([result["radial_peaking_factor"] for result in successful_results], dtype=float)
    keff_values = np.asarray([result["keff"] for result in successful_results], dtype=float)
    rho_values = np.asarray([result["rho_mk"] for result in successful_results], dtype=float)
    failures = len([result for result in unique_results if result.get("status") != "completed"])
    cache_hits = int(all_records["cache_hit"].sum())
    median_time = float(np.median(times)) if times.size else 0.0
    scale_options = [
        {"label": "A", "population": 10, "generations": 10, "elite": 2},
        {"label": "B", "population": 15, "generations": 12, "elite": 2},
        {"label": "C", "population": 20, "generations": 15, "elite": 2},
    ]
    for option in scale_options:
        option["maximum_unique_runs"] = option["population"] + (option["generations"] - 1) * (option["population"] - option["elite"])
        option["estimated_wall_time_seconds"] = max(0, option["maximum_unique_runs"] - 1) * median_time
        option["search_coverage"] = {"A": "低成本初步搜索", "B": "中等覆盖与成本平衡", "C": "最高覆盖、计算成本最大"}[option["label"]]
    # 两代pilot只能验证闭环，不能证明20×15的边际收益；正常且耗时可控时推荐中间规模B。
    if failures == 0 and times.size > 0 and float(np.max(uncertainties)) < 0.10:
        recommended = scale_options[1]
        readiness_status = "ready_for_formal_ga"
        remaining_issues = ["正式规模仍需在论文表4-1和4.4中确认后另行启动。"]
    else:
        recommended = scale_options[0]
        readiness_status = "not_ready_for_formal_ga"
        remaining_issues = ["pilot存在失败、无新计时数据或棒功率统计误差异常。"]
    validation = {
        "implementation_status": "completed", "implemented_functions": [
            "normalize_individual", "validate_individual", "repair_individual", "initialize_population",
            "tournament_select", "set_crossover", "replace_one_mutation", "compute_fitness",
            "evaluate_scheme_real", "create_next_generation", "save_checkpoint", "load_checkpoint",
            "validate_checkpoint_config", "run_genetic_algorithm",
        ],
        "algorithm_tests": algorithm_test_results, "mock_regression_status": "passed_without_old_curve_claim",
        "cache_validation_status": "passed", "checkpoint_validation_status": "passed",
        "seed_strategy": "reference screening=20260730; other schemes use SHA-256(base seed, profile, normalized IDs)",
        "pilot_population": 6, "pilot_generations": 2, "pilot_unique_cases": len(unique_results),
        "new_screening_openmc_runs": int(REAL_GA_RUNTIME_COUNTER["new_screening_openmc_runs"]),
        "cache_hits": cache_hits, "failures": failures,
        "screening_time_mean": float(np.mean(times)), "screening_time_median": median_time,
        "screening_time_max": float(np.max(times)), "pin_uncertainty_mean": float(np.mean(uncertainties)),
        "pilot_Fr_min": float(np.min(fr_values)), "pilot_Fr_max": float(np.max(fr_values)),
        "pilot_keff_min": float(np.min(keff_values)), "pilot_keff_max": float(np.max(keff_values)),
        "pilot_rho_mk_min": float(np.min(rho_values)), "pilot_rho_mk_max": float(np.max(rho_values)),
        "best_pilot_scheme": pilot_run["global_best"]["scheme"],
        "reference_screening_Fr": float(next(result["radial_peaking_factor"] for result in successful_results if tuple(result["normalized_dummy_ids"]) == tuple(REFERENCE_DUMMY_IDS))),
        "formal_scale_options": scale_options,
        "recommended_formal_population": recommended["population"],
        "recommended_formal_generations": recommended["generations"],
        "recommended_formal_elite_size": recommended["elite"],
        "estimated_formal_unique_runs": recommended["maximum_unique_runs"],
        "estimated_formal_wall_time": recommended["estimated_wall_time_seconds"],
        "pilot_total_wall_time": float(pilot_wall_time), "readiness_status": readiness_status,
        "remaining_issues": remaining_issues,
        "formal_ga_started": False, "finalist_review_started": False, "control_rod_inserted_run_count": 0,
    }
    _write_json(GA_PILOT_VALIDATION_PATH, validation)
    pd.DataFrame(scale_options).to_csv(REAL_GA_PILOT_ROOT / "formal_ga_scale_estimates.csv", index=False, encoding="utf-8")
    display(pd.Series({
        "readiness_status": readiness_status, "new_screening_runs": validation["new_screening_openmc_runs"],
        "cache_hits": cache_hits, "failures": failures, "pilot_Fr_min": validation["pilot_Fr_min"],
        "pilot_Fr_max": validation["pilot_Fr_max"], "recommended_population": recommended["population"],
        "recommended_generations": recommended["generations"], "estimated_unique_runs": recommended["maximum_unique_runs"],
    }, name="real GA pilot validation"))
else:
    print("尚未运行pilot，因此不生成或覆盖ga_pilot_validation.json。")

assert RUN_PROFILES["screening"] == {"particles": 5000, "batches": 60, "inactive": 15}
print("pilot单元停止条件确认：本单元不启动正式GA、final复核或控制棒插入计算。")


readiness_status           ready_for_formal_ga
new_screening_runs                           9
cache_hits                                   3
failures                                     0
pilot_Fr_min                          1.203618
pilot_Fr_max                          1.304722
recommended_population                      15
recommended_generations                     12
estimated_unique_runs                      158
Name: real GA pilot validation, dtype: object

停止条件确认：未启动正式GA、final复核或控制棒插入计算。


## 14 正式真实GA与高统计量候选复核

本节复用pilot已验证的离散遗传算子，正式运行20个体、15代的screening搜索。正式checkpoint与pilot完全隔离，但完整物理cache identity一致时允许复用旧screening算例。screening结束后只对全体唯一方案排序得到的前10名进行final复核；参考方案直接读取已有final结果。两个运行开关默认均为False，普通Run All不会启动OpenMC。


In [ ]:
%run -i ./MNSR_formal_ga_workflow.py

formal_workflow_result = None
if RUN_FORMAL_REAL_GA:
    formal_workflow_result = execute_formal_workflow(
        run_finalists=RUN_FINALIST_REVIEW,
        resume=True,
    )
else:
    print("正式GA安全开关关闭；未启动新的OpenMC计算。")
    print("设置环境变量MNSR_RUN_FORMAL_GA=1和MNSR_RUN_FINALISTS=1才会执行本阶段。")


## 15 正式优化结果检查

这里读取已经生成的正式汇总，不触发任何输运计算。`best_verified_scheme`只表示当前物理模型、搜索参数和计算预算下获得的高统计量优势方案，不代表对全部组合的严格全局穷举最优。


In [ ]:
formal_summary_path = Path("build/optimization/real_ga/formal_primary/formal_ga_summary.json")
if formal_summary_path.is_file():
    formal_summary_display = json.loads(formal_summary_path.read_text(encoding="utf-8"))
    display(pd.Series(formal_summary_display, name="formal real GA summary"))
else:
    print("尚无正式GA汇总；安全开关关闭时这是预期状态。")
print("控制棒插入计算次数：0")


## 16 最终候选独立统计确认

本节不重新运行遗传算法，只复用REFERENCE与既有final前5名的replicate 0，并为每个方案增加3个独立final重复。除replicate级统计外，还从四个statepoint逐棒读取原始345项 `kappa-fission` mean/std，按等有效历史权重合并后重新归一化并计算pooled Fr与CV。安全开关默认关闭。


In [ ]:
%run -i ./MNSR_statistical_confirmation.py

statistical_confirmation_result = None
if RUN_STATISTICAL_CONFIRMATION:
    statistical_confirmation_result = execute_statistical_confirmation()
else:
    print("最终候选统计确认安全开关关闭；未启动新的OpenMC计算。")
    print("设置环境变量MNSR_RUN_STAT_CONFIRMATION=1才会执行18个独立确认算例。")


## 17 统计确认结果检查

本单元只读取确认汇总，不触发GA或OpenMC。若状态为 `advantage_but_statistically_weak`，论文只能表述为“在当前统计精度下表现出较优的功率平坦化特征”，不能宣称统计显著优于参考方案。


In [ ]:
confirmation_summary_path = Path("build/optimization/real_ga/statistical_confirmation/statistical_confirmation_summary.json")
if confirmation_summary_path.is_file():
    confirmation_summary_display = json.loads(confirmation_summary_path.read_text(encoding="utf-8"))
    display(pd.Series(confirmation_summary_display, name="statistical confirmation summary"))
else:
    print("尚无统计确认汇总；安全开关关闭时这是预期状态。")
print("GA、控制棒插入、flux mesh和energy spectrum运行次数均为0。")
